In [7]:
!pip install langgraph-checkpoint-sqlite

In [8]:
import sqlite3

Download a ready-made LangGraph checkpoint database
for memory/state persistence demos.


In [9]:
!mkdir -p state_db && [ ! -f state_db/example.db ] && wget -P state_db https://github.com/langchain-ai/langchain-academy/raw/main/module-2/state_db/example.db

**check_same_thread=False is used to allow multiple thread_id's to access the db connection**


In [10]:
db_path="/content/state_db/example.db"
conn=sqlite3.connect(db_path,check_same_thread=False)

In [11]:
cursor=conn.cursor()

In [12]:
from langgraph.checkpoint.sqlite import SqliteSaver

In [13]:
memory=SqliteSaver(conn)

In [14]:
import os
import getpass

if "GROQ_API_KEY" not in os.environ:
  os.environ["GROQ_API_KEY"]=getpass.getpass("Enter api key")

In [4]:
!pip install -q langgraph langchain-groq


In [5]:
# ---------------- INSTALLS ---------------- #



# ---------------- IMPORTS ---------------- #

from typing import Annotated

from langchain_core.messages import (
    HumanMessage,
    SystemMessage
)

from langchain_groq import ChatGroq

from langgraph.graph import (
    StateGraph,
    MessagesState,
    START
)

from langgraph.prebuilt import (
    ToolNode,
    tools_condition
)



# ---------------- TOOLS ---------------- #

def add(a: int, b: int) -> int:
    """Add two numbers"""

    return a + b


def multiply(a: int, b: int) -> int:
    """Multiply two numbers"""

    return a * b


def divide(a: int, b: int) -> float:
    """Divide two numbers"""

    return a / b


tools = [add, multiply, divide]


# ---------------- LLM ---------------- #

llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0
)

llm_with_tools = llm.bind_tools(tools)


# ---------------- SYSTEM MESSAGE ---------------- #

sys_msg = SystemMessage(
    content="""
You are a helpful math assistant.

Use the provided tools whenever needed.
"""
)


# ---------------- NODE ---------------- #

def assistant(state: MessagesState):

    response = llm_with_tools.invoke(
        [sys_msg] + state["messages"]
    )

    return {
        "messages": [response]
    }


# ---------------- BUILD GRAPH ---------------- #

builder = StateGraph(MessagesState)

builder.add_node(
    "assistant",
    assistant
)

builder.add_node(
    "tools",
    ToolNode(tools)
)

builder.add_edge(
    START,
    "assistant"
)

builder.add_conditional_edges(
    "assistant",
    tools_condition
)

builder.add_edge(
    "tools",
    "assistant"
)




content='What is 25 multiplied by 8?' additional_kwargs={} response_metadata={} id='6af9d632-080e-4645-a1d2-36d12a2486e9'

-------------------

content='' additional_kwargs={'tool_calls': [{'id': 'ppfpvwrae', 'function': {'arguments': '{"a":25,"b":8}', 'name': 'multiply'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 19, 'prompt_tokens': 339, 'total_tokens': 358, 'completion_time': 0.066648653, 'completion_tokens_details': None, 'prompt_time': 0.200995212, 'prompt_tokens_details': None, 'queue_time': 1.455370378, 'total_time': 0.267643865}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_dae98b5ecb', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--019e5fbc-a457-7782-9c3a-101bf0ff980d-0' tool_calls=[{'name': 'multiply', 'args': {'a': 25, 'b': 8}, 'id': 'ppfpvwrae', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 339, 'output_tokens': 19, 'to

In [15]:
# ---------------- COMPILE ---------------- #

graph = builder.compile(checkpointer=memory)


In [21]:
config={"configurable":{"thread_id":"67"}}

In [23]:


# ---------------- INVOKE ---------------- #

output = graph.invoke(
    {
        "messages": [
            HumanMessage(
                content="What is 25 multiplied by 8?"
            )
        ]
    },config
)


# ---------------- PRINT ---------------- #

for msg in output["messages"]:
    print(msg)
    print("\n-------------------\n")

output = graph.invoke(
    {
        "messages": [
            HumanMessage(
                content="Im Rahul"
            )
        ]
    },config
)
for msg in output["messages"]:
    print(msg)
    print("\n-------------------\n")

output = graph.invoke(
    {
        "messages": [
            HumanMessage(
                content="whats my name and whats the 1st question i asked"
            )
        ]
    },config
)
print(output["messages"][-1].content)

content='What is 25 multiplied by 8?' additional_kwargs={} response_metadata={} id='d53dae20-f8bc-4280-b432-9e1057ff5d37'

-------------------

content='' additional_kwargs={'tool_calls': [{'id': 'tnfapqr39', 'function': {'arguments': '{"a":25,"b":8}', 'name': 'multiply'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 19, 'prompt_tokens': 339, 'total_tokens': 358, 'completion_time': 0.05920978, 'completion_tokens_details': None, 'prompt_time': 0.017030245, 'prompt_tokens_details': None, 'queue_time': 0.023292734, 'total_time': 0.076240025}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_0761e44d7b', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--019e5fc3-f7e5-71c3-9a4c-12ac8475ac46-0' tool_calls=[{'name': 'multiply', 'args': {'a': 25, 'b': 8}, 'id': 'tnfapqr39', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 339, 'output_tokens': 19, 'tot